<h3>You must download and import this <a href="https://www.kaggle.com/datasets/mldatastudent/league-of-legends-match-data">Kaggle dataset</a>.

<h2>Import requirements

In [1]:
import pandas as pd
import plotly.express as px

<h2>Read in the raw data (per-player)

In [2]:
player_data = pd.read_csv("data/lol_match_data.csv")
print("Number of columns: ", player_data.shape[1])
print("Number of rows: ", player_data.shape[0])
player_data.head(1)

Number of columns:  108
Number of rows:  205110


,match_matchId,match_gameStartTimestamp,match_gameEndTimestamp,match_gameDuration,match_mapId,match_platformId,player_puuid,player_teamId,player_teamPosition,player_lane,...,player_item3_categories,player_item3_priceTotal,player_item4_name,player_item4_description,player_item4_categories,player_item4_priceTotal,player_item5_name,player_item5_description,player_item5_categories,player_item5_priceTotal
0,LA1_1531159804,1.720820e+12,1.720820e+12,1834.0,11.0,LA1,QPstXBo4FWSoly8yTtzmHFjsgtwUJrVzRhFTWlO3irBaEd...,blue,TOP,JUNGLE,...,"['Health', 'Damage', 'CooldownReduction', 'Abi...",3100,Sterak's Gage,<mainText><stats><attention> 400</attention> H...,"['Health', 'Damage', 'Tenacity']",3200,Stealth Ward,<mainText><stats></stats><br><br> <active>ACTI...,"['Active', 'Jungle', 'Lane', 'Trinket', 'Vision']",0


<p>This data is gathered from high-rank lobbies, which contain a small pool of re-ocurring players. Each player may appear more than once in the dataset, on different teams during different matches. Therefore, the number of rows != the number of unique players (205,110 vs. 24,279). Each game contains 10 players, so we know there are 20,511 total games included.

In [3]:
print("Number of unique players: ", player_data["player_puuid"].nunique())
print("Number of unique teams: ", len(player_data[["match_matchId", "player_teamId"]].drop_duplicates()))
print("Number of unique matches: ", player_data["match_matchId"].nunique())

Number of unique players:  24279
Number of unique teams:  41022
Number of unique matches:  20511


<h2>Clean and aggregate the data (per-team)

In [4]:
# make new unique match-team ids to avoid needing to use both columns later
player_data["match_teamId"] = player_data["match_matchId"] + player_data["player_teamId"]

<p>We only want to keep columns that we will use as features in our model. We can remove unnecessary columns related to player accounts, perks and items. The "player_win" column will eventually be our label (this tells us whether this team won the game).

In [5]:
features_of_interest = ["match_teamId", "match_gameDuration", "player_teamPosition", "player_win",
                          "player_kills", "player_deaths", "player_assists", "player_goldEarned", "player_visionScore", "player_damageDealtToTurrets",
                          "team_baron_kills", "team_dragon_kills", "team_riftHerald_kills", "team_tower_kills"]
player_data = player_data[features_of_interest]

<p>Below you can see two teams from one game, notice the ten different players, with five roles per team. Player-specific stats such as "player_kills" vary per row while team stats such as "player_win" and "team_dragon_kills" are the same.

In [6]:
player_data.head(10)

,match_teamId,match_gameDuration,player_teamPosition,player_win,player_kills,player_deaths,player_assists,player_goldEarned,player_visionScore,player_damageDealtToTurrets,team_baron_kills,team_dragon_kills,team_riftHerald_kills,team_tower_kills
0,LA1_1531159804blue,1834.0,TOP,True,14.0,4.0,5.0,15613.0,29.0,14499.0,1.0,3.0,0.0,7.0
1,LA1_1531159804blue,1834.0,JUNGLE,True,2.0,4.0,17.0,10279.0,28.0,990.0,1.0,3.0,0.0,7.0
2,LA1_1531159804blue,1834.0,MIDDLE,True,2.0,6.0,16.0,10314.0,20.0,765.0,1.0,3.0,0.0,7.0
3,LA1_1531159804blue,1834.0,BOTTOM,True,19.0,7.0,10.0,17195.0,23.0,4540.0,1.0,3.0,0.0,7.0
4,LA1_1531159804blue,1834.0,UTILITY,True,2.0,5.0,24.0,9350.0,87.0,487.0,1.0,3.0,0.0,7.0
5,LA1_1531159804red,1834.0,TOP,False,8.0,7.0,3.0,16804.0,30.0,14107.0,1.0,1.0,1.0,7.0
6,LA1_1531159804red,1834.0,JUNGLE,False,10.0,6.0,5.0,13346.0,36.0,490.0,1.0,1.0,1.0,7.0
7,LA1_1531159804red,1834.0,MIDDLE,False,2.0,6.0,8.0,10616.0,21.0,2159.0,1.0,1.0,1.0,7.0
8,LA1_1531159804red,1834.0,BOTTOM,False,4.0,11.0,7.0,10196.0,24.0,3292.0,1.0,1.0,1.0,7.0
9,LA1_1531159804red,1834.0,UTILITY,False,2.0,9.0,18.0,9609.0,83.0,771.0,1.0,1.0,1.0,7.0


<h4>Isolate position-specific data

In [7]:
top = player_data[player_data["player_teamPosition"] == "TOP"]
jungle = player_data[player_data["player_teamPosition"] == "JUNGLE"]
middle = player_data[player_data["player_teamPosition"] == "MIDDLE"]
bottom = player_data[player_data["player_teamPosition"] == "BOTTOM"]
support = player_data[player_data["player_teamPosition"] == "UTILITY"]

<h4>Aggregate team stats by match

In [8]:
matches_df = player_data[["match_teamId"]]
matches_df = matches_df.drop_duplicates().reset_index().drop(columns=["index"])
matches_df.head(2)

,match_teamId
0,LA1_1531159804blue
1,LA1_1531159804red


In [9]:
# save match duration
row_per_team = player_data.drop_duplicates(subset="match_teamId")
matches_df["match_duration"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["match_gameDuration"])


# save each role's stats
roles = {"top": top, "jg": jungle, "mid": middle, "bot": bottom, "sup": support}
for role in roles:
    matches_df = matches_df.merge(roles[role][["match_teamId", "player_kills", "player_deaths", "player_assists", "player_goldEarned", "player_visionScore", "player_damageDealtToTurrets"]], on="match_teamId", how="left"
                              ).rename(columns={"player_kills": f"{role}_kills", "player_deaths": f"{role}_deaths", "player_assists": f"{role}_assists", "player_goldEarned": f"{role}_gold", "player_visionScore": f"{role}_vision", "player_damageDealtToTurrets": f"{role}_tower_damage"})

# save team stats
matches_df["team_baron_kills"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["team_baron_kills"])
matches_df["team_dragon_kills"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["team_dragon_kills"])
matches_df["team_riftHerald_kills"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["team_riftHerald_kills"])
matches_df["team_tower_kills"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["team_tower_kills"])

# add new, aggregate team stats
team_kills = player_data.groupby("match_teamId")["player_kills"].sum()
team_deaths = player_data.groupby("match_teamId")["player_deaths"].sum()
team_assists = player_data.groupby("match_teamId")["player_assists"].sum()
team_gold = player_data.groupby("match_teamId")["player_goldEarned"].sum()
team_vision = player_data.groupby("match_teamId")["player_visionScore"].sum()
team_tower_damage = player_data.groupby("match_teamId")["player_damageDealtToTurrets"].sum()
matches_df["team_kills"] = matches_df["match_teamId"].map(team_kills)
matches_df["team_deaths"] = matches_df["match_teamId"].map(team_deaths)
matches_df["team_assists"] = matches_df["match_teamId"].map(team_assists)
matches_df["team_gold"] = matches_df["match_teamId"].map(team_gold)
matches_df["team_vision"] = matches_df["match_teamId"].map(team_vision)
matches_df["team_tower_damage"] = matches_df["match_teamId"].map(team_tower_damage)

# save match outcome
matches_df["team_win"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["player_win"])

<h2>Addressing "early forfeit" matches

<p>Normally, a team can only surrender once a game has reached 15 minutes in duration. Early forfeit matches are those where a team surrenders before 15 minutes (900 seconds). These matches end early because someone has left the game, and aren't representative of a normal match, so we want to remove them.

In [10]:
early_forfeit_matches_df = matches_df[matches_df["match_duration"] <= 900]
print("Number of early forfeit matches: ", early_forfeit_matches_df.shape[0])
early_forfeit_matches_df.head(2)

Number of early forfeit matches:  848


,match_teamId,match_duration,top_kills,top_deaths,top_assists,top_gold,top_vision,top_tower_damage,jg_kills,jg_deaths,...,team_dragon_kills,team_riftHerald_kills,team_tower_kills,team_kills,team_deaths,team_assists,team_gold,team_vision,team_tower_damage,team_win
16,LA1_1530878057blue,285.0,1.0,2.0,0.0,1460.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,3.0,4.0,4.0,7235.0,3.0,0.0,False
17,LA1_1530878057red,285.0,2.0,1.0,0.0,1945.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,4.0,3.0,2.0,8414.0,6.0,354.0,True


In [11]:
matches_df = matches_df[matches_df["match_duration"] > 900]
print("Number of teams in final dataset: ", matches_df.shape[0])
print("Number of matches in final dataset: ", matches_df.shape[0] // 2)

Number of teams in final dataset:  40174
Number of matches in final dataset:  20087


In [12]:
features = list(matches_df.columns)
# print("Number of features in the final dataset: ", len(features))
print(features[0:3])
print(features[3:9])
print(features[9:15])
print(features[15:21])
print(features[21:27])
print(features[27:33])
print(features[33:39])

['match_teamId', 'match_duration', 'top_kills']
['top_deaths', 'top_assists', 'top_gold', 'top_vision', 'top_tower_damage', 'jg_kills']
['jg_deaths', 'jg_assists', 'jg_gold', 'jg_vision', 'jg_tower_damage', 'mid_kills']
['mid_deaths', 'mid_assists', 'mid_gold', 'mid_vision', 'mid_tower_damage', 'bot_kills']
['bot_deaths', 'bot_assists', 'bot_gold', 'bot_vision', 'bot_tower_damage', 'sup_kills']
['sup_deaths', 'sup_assists', 'sup_gold', 'sup_vision', 'sup_tower_damage', 'team_baron_kills']
['team_dragon_kills', 'team_riftHerald_kills', 'team_tower_kills', 'team_kills', 'team_deaths', 'team_assists']


<p>Now below we can view four teams from two different games. Can we tell just by the feature values which team has TRUE for "team_win"? Look at the "team_turrets_killed" feature.

In [13]:
matches_df.head(2)

,match_teamId,match_duration,top_kills,top_deaths,top_assists,top_gold,top_vision,top_tower_damage,jg_kills,jg_deaths,...,team_dragon_kills,team_riftHerald_kills,team_tower_kills,team_kills,team_deaths,team_assists,team_gold,team_vision,team_tower_damage,team_win
0,LA1_1531159804blue,1834.0,14.0,4.0,5.0,15613.0,29.0,14499.0,2.0,4.0,...,3.0,0.0,7.0,39.0,26.0,72.0,62751.0,187.0,21281.0,True
1,LA1_1531159804red,1834.0,8.0,7.0,3.0,16804.0,30.0,14107.0,10.0,6.0,...,1.0,1.0,7.0,26.0,39.0,41.0,60571.0,194.0,20819.0,False


<h2>Should "team_tower_kills" be included as a feature?

<h4>Let's look at the distribution of the "top_kills" feature between win and loss matches.

<p>In general, this statistic is seen as a very big indicator of who wins a game, because whoever wins top lane can easily take objectives.

In [14]:
fig = px.box(matches_df, x="team_win", y="top_kills")
fig.write_image("images/TopKills_by_TeamWin.png")
fig.show()

<p>As shown by the graph, there is a difference for this feature between win and loss, but the median only differs by 2.

<h4>Now, let's look at the "team_tower_kills" feature between win and loss matches.

In [15]:
fig = px.box(matches_df, x="team_win", y="team_tower_kills")
fig.write_image("images/TeamTowerKills_by_TeamWin.png")
fig.show()

<p>As shown here, the median number of towers killed for win matches is the upper fence for loss matches, meaning it is definitely a "giveaway" of match results.

<h4>Is any other feature closely tied to "team_tower_kills"?

In [16]:
fig = px.scatter(matches_df, x="team_tower_kills", y="team_gold")
fig.write_image("images/TeamGold_by_TeamTowerKills.png")
fig.show()

<p>Looking at this, the feature "team_gold" has a positive relationship with "team_tower_kills", which makes sense. In this way, removing the "team_tower_kills" feature will not entirely remove this information because it is closely tied to other stats.

In [17]:
fig = px.scatter(matches_df, x="team_tower_kills", y="team_tower_damage")
fig.write_image("images/TopTowerDamage_by_TeamTowerKills.png")
fig.show()

<p>The same can be said for building damage features, such as "top_building_damage", which also makes sense.

In [18]:
fig = px.box(matches_df, x="team_win", y="team_tower_damage")
fig.write_image("images/TeamTowerDamage_by_TeamWin.png")
fig.show()

<p>Looking closer at this feature, it has a similar win / loss match breakdown as "team_tower_kills", but maybe less extreme on each end.

<h4>Remove "team_tower_kills" feature

In [19]:
matches_df = matches_df.drop(columns=["team_tower_kills"])
matches_df.head(2)

,match_teamId,match_duration,top_kills,top_deaths,top_assists,top_gold,top_vision,top_tower_damage,jg_kills,jg_deaths,...,team_baron_kills,team_dragon_kills,team_riftHerald_kills,team_kills,team_deaths,team_assists,team_gold,team_vision,team_tower_damage,team_win
0,LA1_1531159804blue,1834.0,14.0,4.0,5.0,15613.0,29.0,14499.0,2.0,4.0,...,1.0,3.0,0.0,39.0,26.0,72.0,62751.0,187.0,21281.0,True
1,LA1_1531159804red,1834.0,8.0,7.0,3.0,16804.0,30.0,14107.0,10.0,6.0,...,1.0,1.0,1.0,26.0,39.0,41.0,60571.0,194.0,20819.0,False


<h4>Other interesting visualizations

In [20]:
fig = px.scatter(matches_df, x="match_duration", y="team_gold", color="team_win")
fig.write_image("images/TeamGold_by_MatchDuration.png")
fig.show()

<h2>Save final cleaned dataset

<h4>Convert numbers to integer type and booleans to 0 / 1

In [21]:
matches_df[list(matches_df.columns)[1:]] = matches_df[list(matches_df.columns)[1:]].apply(pd.to_numeric, errors='coerce').astype("Int64")
matches_df[list(matches_df.columns)[1:]] = matches_df[list(matches_df.columns)[1:]].fillna(0)

<h4>Export to csv

In [22]:
matches_df = matches_df.drop(columns=["match_teamId"])
matches_df.to_csv('data/lol_dataset.csv', index=False)

<h2>Save normalized dataset

<h4>All player match statistics are heavily influenced by match_duration, so we should normalize by this number to get normalized stats (per minute)

<h4>Divide numeric stat columns by match duration

In [23]:
matches_df.head(1)

,match_duration,top_kills,top_deaths,top_assists,top_gold,top_vision,top_tower_damage,jg_kills,jg_deaths,jg_assists,...,team_baron_kills,team_dragon_kills,team_riftHerald_kills,team_kills,team_deaths,team_assists,team_gold,team_vision,team_tower_damage,team_win
0,1834,14,4,5,15613,29,14499,2,4,17,...,1,3,0,39,26,72,62751,187,21281,1


In [24]:
cols_to_fix = matches_df.columns[1:-10].append(matches_df.columns[-7:-1])
matches_df[cols_to_fix] = matches_df[cols_to_fix].div(matches_df['match_duration'] / 60, axis=0) # matches_df["match_duration"] / 60 gives us the number of minutes
matches_df.head(1)

,match_duration,top_kills,top_deaths,top_assists,top_gold,top_vision,top_tower_damage,jg_kills,jg_deaths,jg_assists,...,team_baron_kills,team_dragon_kills,team_riftHerald_kills,team_kills,team_deaths,team_assists,team_gold,team_vision,team_tower_damage,team_win
0,1834,0.458015,0.130862,0.163577,510.785169,0.948746,474.34024,0.065431,0.130862,0.556161,...,1,3,0,1.2759,0.8506,2.355507,2052.922574,6.117775,696.215921,1


<h4>Rename effected columns

In [25]:
matches_df = matches_df.rename(columns={c: c + '_per_min' for c in cols_to_fix})

In [26]:
matches_df = matches_df.drop(columns=["match_duration"]) # this column isn't important if other columns are normalized, right?
matches_df.to_csv('data/lol_dataset_normalized.csv', index=False)